# 03 - Model Training
ConvLSTM for spatiotemporal fishing ground prediction

In [1]:
!pip install -q tensorflow

In [2]:
from google.colab import drive
import xarray as xr
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import ConvLSTM2D, Conv2D, BatchNormalization, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

drive.mount('/content/drive')
DATA_DIR = "/content/drive/MyDrive/fishing_project/"

Mounted at /content/drive


In [4]:
# Load preprocessed data
features = xr.open_dataset(DATA_DIR + 'preprocessed_features.nc')
print(f"Features: {dict(features.dims)}")
print(f"Variables: {list(features.data_vars)}")
# Expected: ['chl', 'nppv', 'ssh', 'sst', 'uo', 'vo', 'fishing_effort']

Features: {'time': 72, 'latitude': 41, 'longitude': 25}
Variables: ['chl', 'nppv', 'ssh', 'sst', 'uo', 'vo', 'fishing_effort']


/tmp/ipykernel_623/2999282416.py:3: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"Features: {dict(features.dims)}")


In [5]:
# Create sequences (3 months input → 1 month output)
SEQ_LEN  = 3
PRED_LEN = 1

# Stack all 7 feature channels: shape (time, lat, lon, 7)
X_stack = np.stack([
    features['sst'].values,            # Ch 0 - Sea surface temperature
    features['ssh'].values,            # Ch 1 - Sea surface height
    features['vo'].values,             # Ch 2 - Northward velocity
    features['uo'].values,             # Ch 3 - Eastward velocity
    features['chl'].values,            # Ch 4 - Chlorophyll-a
    features['nppv'].values,           # Ch 5 - Net primary production
    features['fishing_effort'].values, # Ch 6 - AIS fishing effort (log-norm)
], axis=-1)

# Target: predict fishing effort at the next time step
n_months = len(features.time)
n_lat    = len(features.latitude)
n_lon    = len(features.longitude)

y_array = features['fishing_effort'].values  # (n_months, n_lat, n_lon)

# Replace NaNs (land/masked cells) with 0
X_stack = np.nan_to_num(X_stack, nan=0.0)
y_array = np.nan_to_num(y_array, nan=0.0)

print(f"X shape: {X_stack.shape}")  # (72, 41, 25, 7)
print(f"y shape: {y_array.shape}")  # (72, 41, 25)


X shape: (72, 41, 25, 7)
y shape: (72, 41, 25)


In [6]:
# Generate sequences
def create_sequences(X, y, seq_len, pred_len):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_len - pred_len + 1):
        X_seq.append(X[i:i+seq_len])
        y_seq.append(y[i+seq_len:i+seq_len+pred_len])
    return np.array(X_seq), np.array(y_seq)

X_seq, y_seq = create_sequences(X_stack, y_array, SEQ_LEN, PRED_LEN)
print(f"Sequences: {X_seq.shape} → {y_seq.shape}")

Sequences: (69, 3, 41, 25, 7) → (69, 1, 41, 25)


In [7]:
# Train/val/test split (70/15/15)
train_size = int(0.7 * len(X_seq))
val_size = int(0.15 * len(X_seq))

X_train = X_seq[:train_size]
y_train = y_seq[:train_size]
X_val = X_seq[train_size:train_size+val_size]
y_val = y_seq[train_size:train_size+val_size]
X_test = X_seq[train_size+val_size:]
y_test = y_seq[train_size+val_size:]

print(f"Train: {X_train.shape}")
print(f"Val: {X_val.shape}")
print(f"Test: {X_test.shape}")

Train: (48, 3, 41, 25, 7)
Val: (10, 3, 41, 25, 7)
Test: (11, 3, 41, 25, 7)


In [8]:
# Build ConvLSTM model
model = Sequential([
    ConvLSTM2D(
        filters=64,
        kernel_size=(3, 3),
        padding='same',
        return_sequences=True,
        input_shape=(SEQ_LEN, n_lat, n_lon, 7)
    ),
    BatchNormalization(),
    Dropout(0.2),

    ConvLSTM2D(
        filters=32,
        kernel_size=(3, 3),
        padding='same',
        return_sequences=False
    ),
    BatchNormalization(),
    Dropout(0.2),

    Conv2D(filters=1, kernel_size=(1, 1), activation='sigmoid', padding='same')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['mae']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv_lstm2d (ConvLSTM2D)        │ (None, 3, 41, 25, 64)  │       163,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 3, 41, 25, 64)  │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 3, 41, 25, 64)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_lstm2d_1 (ConvLSTM2D)      │ (None, 41, 25, 32)     │       110,720 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 41, 25, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 41, 25, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 41, 25, 1)      │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 274,977 (1.05 MB)

 Trainable params: 274,785 (1.05 MB)

 Non-trainable params: 192 (768.00 B)

In [9]:
# Callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ModelCheckpoint(DATA_DIR + 'best_model.h5', monitor='val_loss', save_best_only=True)
]

In [10]:
# Reshape y data to match model output: (samples, lat, lon, 1)
y_train = y_train.squeeze(axis=1)[..., np.newaxis]  # (samples, 1, lat, lon) → (samples, lat, lon, 1)
y_val = y_val.squeeze(axis=1)[..., np.newaxis]
y_test = y_test.squeeze(axis=1)[..., np.newaxis]

print(f"Reshaped y_train: {y_train.shape}")
print(f"Reshaped y_val: {y_val.shape}")
print(f"Reshaped y_test: {y_test.shape}")

# Train model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=8,
    callbacks=callbacks,
    verbose=1
)

Reshaped y_train: (48, 41, 25, 1)
Reshaped y_val: (10, 41, 25, 1)
Reshaped y_test: (11, 41, 25, 1)
Epoch 1/50
5/6 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 1.0005 - mae: 0.4874

6/6 ━━━━━━━━━━━━━━━━━━━━ 15s 641ms/step - loss: 0.9048 - mae: 0.4799 - val_loss: 0.6524 - val_mae: 0.3787
Epoch 2/50
5/6 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 0.7623 - mae: 0.4698

6/6 ━━━━━━━━━━━━━━━━━━━━ 9s 66ms/step - loss: 0.7494 - mae: 0.4629 - val_loss: 0.6310 - val_mae: 0.3663
Epoch 3/50
5/6 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 0.7185 - mae: 0.4590

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - loss: 0.7067 - mae: 0.4507 - val_loss: 0.6118 - val_mae: 0.3552
Epoch 4/50
5/6 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 0.6899 - mae: 0.4383

6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 55ms/step - loss: 0.6828 - mae: 0.4390 - val_loss: 0.5963 - val_mae: 0.3461
Epoch 5/50
5/6 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 0.6594 - mae: 0.4394

...

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - loss: 0.1719 - mae: 0.1204 - val_loss: 0.3319 - val_mae: 0.1536
Epoch 38/50
4/6 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.1669 - mae: 0.1150 

6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 60ms/step - loss: 0.1635 - mae: 0.1124 - val_loss: 0.3306 - val_mae: 0.1504
Epoch 39/50
4/6 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.1524 - mae: 0.1091

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - loss: 0.1555 - mae: 0.1067 - val_loss: 0.3299 - val_mae: 0.1478
Epoch 40/50
4/6 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.1570 - mae: 0.1029

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - loss: 0.1511 - mae: 0.1026 - val_loss: 0.3297 - val_mae: 0.1457
Epoch 41/50
4/6 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.1537 - mae: 0.1000

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - loss: 0.1437 - mae: 0.0968 - val_loss: 0.3295 - val_mae: 0.1429
Epoch 42/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 0.1376 - mae: 0.0914 - val_loss: 0.3316 - val_mae: 0.1418
Epoch 43/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 0.1311 - mae: 0.0865 - val_loss: 0.3323 - val_mae: 0.1389
Epoch 44/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 0.1266 - mae: 0.0815 - val_loss: 0.3320 - val_mae: 0.1373
Epoch 45/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 0.1233 - mae: 0.0790 - val_loss: 0.3318 - val_mae: 0.1374
Epoch 46/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 0.1190 - mae: 0.0753 - val_loss: 0.3299 - val_mae: 0.1346
Epoch 47/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 0.1163 - mae: 0.0726 - val_loss: 0.3341 - val_mae: 0.1331
Epoch 48/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 0.1147 - mae: 0.0711 - val_loss: 0.3373 - val_mae: 0.1282
Epoch 49/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 0.1130 - mae: 0.0687 - val

In [11]:
# Save final model
model.save(DATA_DIR + 'convlstm_model.h5')
# Save training history
import json
with open(DATA_DIR + 'training_history.json', 'w') as f:
    json.dump(history.history, f)
# Save test data for evaluation notebook
np.save(DATA_DIR + 'X_test.npy', X_test)
np.save(DATA_DIR + 'y_test.npy', y_test)
print("\n✔ Model training complete")


✅ Model training complete
